In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################
start_time = time.time();

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
job_array=False;index_adjust=0
ocean_fraction=2/8

# # dx = 1 km; Np = 1M; Nt = 5 min
# data1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc', decode_timedelta=True) #***
# parcel1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc', decode_timedelta=True) #***
# res='1km';t_res='5min'
# Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min.nc', decode_timedelta=True) #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc', decode_timedelta=True) #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 50M; Nz = 95
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min_95nz.nc', decode_timedelta=True) #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_95nz.nc', decode_timedelta=True) #***
# res='1km'; t_res='1min_95nz'; Np_str='50e6'

# dx = 250m; Np = 50M
#Importing Model Data
dir2='/home/air673/koa_scratch/'
data1=xr.open_dataset(dir2+'cm1out_250m_1min_50M.nc', decode_timedelta=True) #***
parcel1=xr.open_dataset(dir2+'cm1out_pdata_250m_1min_50M.nc', decode_timedelta=True) #***
res='250m'; t_res='1min'; Np_str='50e6'

In [2]:
############################################################################################
#MODEL AND ALGORITHM NUMERICAL PARAMETERS
times=data1['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
kms=np.argmax(data1['xh'].values-data1['xh'][0].values >= 1) #finds how many x grids is 1 km

In [3]:
###################################################################################################################################
#PLOTTING
plotting=False #KEEP FALSE WHEN RUNNING
plotting=True

In [4]:
#TIME 00:00:00 Function
#Gets the realtime for the current timestep
def get_time(t):
    # init_day,init_hour,init_min=0,0,0
    init_day,init_hour,init_min=0,6,0
    times=data1['time'].values/(1e9 * 60); time_inc=times.astype(int)[1]-times.astype(int)[0]
    current_min=init_hour*60+init_min+time_inc*t;
    
    days = init_day + (current_min // (24 * 60))
    
    remain_min = (init_min+time_inc*t) % (24 * 60); 
    hours = (init_hour + (remain_min // 60)) % 24
    mins = remain_min % 60

    ##############################################
    days=str(days);hours=str(hours);mins=str(mins)
    if len(days)==1:days='0'+days
    if len(hours)==1:hours='0'+hours
    if len(mins)==1:mins='0'+mins
    ##############################################

    combo=days+":"+hours+":"+mins
    return days,hours,mins

In [5]:
#Get Data Functions
def get_conv_plot(t,zlev):
    import h5py
    # print('calculating convergence and taking mean')
    if res=='1km':
        dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
    elif res=='250m':
        dir2='/home/air673/koa_scratch/'
    file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{t_res}.h5'
    with h5py.File(file_path, 'r') as f:
        Conv = f['conv'][t,zlev,:,:] #*#*#* For JobArray
    return Conv

In [6]:
#GETTING SOME INDICES
# t=350;yind=400
t=100;yind=100
zlev=np.where(data1['zh']>=280/1e3)[0][0].item() #(zf first level >= ~0.28 km)
t,yind,zlev

(100, 100, 6)

In [7]:
#LOADING SOME DATA

folder_path = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_Tracking_Out/Plots/' #plot image output foldername

load_dir = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/'

open_name = load_dir+f'whereCL_{res}_{t_res}_ALL_CLS.nc'
# whereCL_ALL_CLS = xr.open_dataset(open_name, chunks=dict(time=1, z=-1, y=-1,x=-1))['maxconv_x']
whereCL_ALL_CLS = xr.open_dataset(open_name)['maxconv_x']

In [ ]:
whereCL_ALL_CLS

In [ ]:
# local_maxes=whereCL_ALL_CLS.isel(time=t, z=zlev, y=yind, x=slice(0, 500)).values
local_maxes=whereCL_ALL_CLS.isel(time=t, z=zlev, y=yind).values
local_maxes

In [7]:
# #DATA LOADING ISSUE HERE ***
# conv_z=get_conv_plot(t,zlev)

In [9]:
folder_path = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_Tracking_Out/Plots/' #plot image output foldername

load_dir = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/'

open_name = load_dir+f'whereCL_{res}_{t_res}_ALL_CLS.nc'
# whereCL_ALL_CLS = xr.open_dataset(open_name, chunks=dict(time=1, z=-1, y=-1,x=-1))['maxconv_x']
whereCL_ALL_CLS = xr.open_dataset(open_name)['maxconv_x']

# open_name = load_dir+f'whereCL_{res}_{t_res}_{Np_str}_ONLY_SBZS.nc'
# whereCL_ONLY_SBZS = xr.open_dataset(open_name, chunks=dict(time=1, z=-1, y=-1,x=-1))['maxconv_x']
# # whereCL_ONLY_SBZS=xr.open_dataset(open_name)['maxconv_x'].data

def CL_plotting(t,z_lev,ONLY_SBZ,SAVING):
    if np.mod(t,10)==0: print(f'current time step: {t}/{len(data1["time"])}')
    dx=np.round(data1['xf'][1]-data1['xf'][0],2).item();dy=dx #gets the dx,dy in meters (0.25m in this case)
    
    
    #Taking Convergence of current timestep and zlevel
    zlev=np.where(data1['zh']>=280/1e3)[0][0] #(zf first level >= ~0.28 km)
    conv_z=get_conv_plot(t,zlev)

    #Plotting horizontal layer at ~0.28 km 
    channel_aspect_ratio = 5
    figwidth=20
    plt.figure(figsize=(figwidth, figwidth/channel_aspect_ratio)) 
    contour=plt.contourf(conv_z*1000)#,levels=np.arange(-1.5,1.5,.01),vmin=vmin,vmax=vmax)
    
    colorbar = plt.colorbar(contour,label=f'{dx*1000:.0f}'+r'$\ s^{-1}$')# ,pad=pad)
    #####################################################################
    #Gets time for Plot Label
    [days,hours,mins]=get_time(t+index_adjust)
    value=data1['zf'][zlev].values; 
    plt.title(f'Convergence at t={t+index_adjust}={days}:{hours}:{mins}, z={value:.2f} km')
    plt.xlabel('x (km)')
    plt.ylabel('y (km)')
    #####################################################################

    # plt.contourf(data1['pwat'].isel(time=t),alpha=0.3,cmap='Reds') #rain, prate, pwat #TESTING

    #ADDING TRACKED LOCATION SCATTER
    #for each ylevel, apply FindLocalMaxes Function
    for yind in range(0,len(data1['yh'])): #plot maximums for each row
        if ONLY_SBZ==False:
            local_maxes=whereCL_ALL_CLS.isel(time=t, z=z_lev, y=yind).values
            local_maxes=local_maxes[local_maxes != -1]
        elif ONLY_SBZ==True:
            local_maxes=whereCL_ONLY_SBZS.isel(time=t, z=z_lev, y=yind).values
            local_maxes=local_maxes[local_maxes != -1]
        plt.scatter(local_maxes,np.array([yind]*len(local_maxes)),color='red',s=1)

    # SAVING PLOT
    if SAVING==True:
        if ONLY_SBZ==False:
            plt.savefig(os.path.join(folder_path, f"ALL_CLS/plot_{t+index_adjust}.png"),dpi=72) #save the plot
        if ONLY_SBZ==True:
            plt.savefig(os.path.join(folder_path, f"ONLY_SBZS/plot_{t+index_adjust}.png"),dpi=72) #save the plot
        # print('finished saving plot') #TESTING
        plt.close()

ERROR! Session/line number was not unique in database. History logging moved to new session 5778


In [ ]:
# ####PLOT FORM /FIGURES.ipynb
# def CL_plotting(t, ax, ONLY_SBZ, font_size=12, index_adjust=0):
#     if np.mod(t, 10) == 0:
#         print(f'Current time step: {t}/{len(data["time"])}')

#     zlev = 3
#     print('Loading Data')
#     conv_z = get_conv(t, zlev)
#     print('Plotting')

#     dx = np.round(data['xf'][1] - data['xf'][0], 2).item()
#     dy = dx

#     xh = data['xh'] - data['xh'][0]
#     yh = data['yh'] - data['yh'][0]

#     contour = ax.contourf(xh, yh, conv_z * 1000, levels=40)
#     cbar = plt.colorbar(contour, ax=ax, pad=0)
#     # cbar.set_label(f'{dx * 1000:.0f}' + r'$\ s^{-1}$', fontsize=font_size)
#     # cbar.set_label(r'$-\nabla \cdot \mathbf{V}_H\ (s^{-1}$)', fontsize=font_size)
#     cbar.set_label(r'$-\nabla \cdot \vec{V}_H\ (s^{-1})$', fontsize=font_size)
#     cbar.ax.tick_params(labelsize=font_size)
#     cbar.ax.yaxis.label.set_size(font_size)

#     ([days, hours, mins], _) = get_time(data, t + index_adjust, (0, 6, 0))
#     z_value = data['zf'][zlev].values * 1000

#     # ax.set_title(f'Horizontal Convergence at t = {t + index_adjust} = {days}:{hours}:{mins}, z = {z_value:.0f} m\nWith Tracked Convergence Local Y-Maxima Overlayed',fontsize=font_size)
#     ax.set_xlabel('x (km)', fontsize=font_size)
#     ax.set_ylabel('y (km)', fontsize=font_size)
#     ax.tick_params(axis='both', which='major', labelsize=font_size)

#     # Scatter max convergence points
#     for yind in range(len(data['yh'])):
#         if ONLY_SBZ:
#             local_maxes = whereCL_ONLY_SBZS[t, 3, yind].data
#         else:
#             local_maxes = whereCL_ALL_CLS[t, 3, yind].data
#         local_maxes = local_maxes[local_maxes != -1]
#         ax.scatter(local_maxes, [yind] * len(local_maxes), color='red', s=1)

#     # Coastline
#     ax.axvline(x=len(data['xf']) * ocean_fraction, color='black', linewidth=1.5, label='Coastline')

#     # Legend
#     handle_pts = mlines.Line2D([], [], color='red', marker='o', linestyle='None', markersize=6, label='Convergence Local Y-Maxima')
#     handle_time = mlines.Line2D([], [], color='none', label=f't = {t + index_adjust} = {days}:{hours}:{mins}')
#     handle_z = mlines.Line2D([], [], color='none', label=f'z = {data["zh"][zlev].item() * 1000:.0f} m')
#     handle_c = mlines.Line2D([], [], color='black', lw=3, label='Coastline')
#     legend = ax.legend(handles=[handle_pts, handle_c, handle_time, handle_z], loc='upper left', fontsize=font_size)
#     for text in legend.get_texts():
#         text.set_fontsize(font_size)


# # # Example call
# # t = 100 if t_res == '5min' else 100 * 5
# # channel_aspect_ratio = 5
# # figwidth = 20; dpi = 300
# # fig, axes = plt.subplots(nrows=1,ncols=1,figsize=(figwidth, figwidth / channel_aspect_ratio), dpi=dpi)
# # CL_plotting(t, axes, ONLY_SBZ=False)

# # # # Save figure
# # # plt.savefig(os.path.join(folder_path, f"convergence.jpg"),dpi=dpi) 

In [ ]:
if plotting==True:
    #TESTING INDIVIDUAL PLOTS
    #########################
    t=100;#t=360
    # t+=20
    z_lev=3;#z_lev=7
    CL_plotting(t=t,z_lev=z_lev,ONLY_SBZ=False,SAVING=False)

In [31]:
if plotting==True:
    #OUTPUTTING ALL PLOTS
    #####################
    for t in np.arange(len(data1['time'])):
        CL_plotting(t=t,z_lev=z_lev,ONLY_SBZ=False,SAVING=True)

current time step: 0/661
current time step: 10/661
current time step: 20/661
current time step: 30/661
current time step: 40/661
current time step: 50/661
current time step: 60/661
current time step: 70/661
current time step: 80/661
current time step: 90/661
current time step: 100/661
current time step: 110/661
current time step: 120/661
current time step: 130/661
current time step: 140/661
current time step: 150/661
current time step: 160/661
current time step: 170/661
current time step: 180/661
current time step: 190/661
current time step: 200/661
current time step: 210/661
current time step: 220/661
current time step: 230/661
current time step: 240/661
current time step: 250/661
current time step: 260/661
current time step: 270/661
current time step: 280/661
current time step: 290/661
current time step: 300/661
current time step: 310/661
current time step: 320/661
current time step: 330/661
current time step: 340/661
current time step: 350/661
current time step: 360/661
current time

In [32]:
if plotting==True:
    #MAKE A GIF OF ALL TRACKED IMAGES
    #####################
    from PIL import Image
    import os
    
    ONLY_SBZ=False
    # ONLY_SBZ=True
    # Folder containing the pictures
    if ONLY_SBZ==False:
        input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_Tracking_Out/Plots/ALL_CLS/'
    if ONLY_SBZ==True:
        input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_Tracking_Out/Plots/ONLY_SBZS/'
    
    image_filenames = sorted([filename for filename in os.listdir(input_folder) if filename.endswith((".jpg", ".png"))], 
                             key=lambda x: int(x.split('_')[-1].split('.')[0]))
    
    # List to store image objects
    images = []
    
    # Iterate over all files in the folder
    for filename in image_filenames:
        # Check if the file is an image
        if (filename.endswith(".jpg") or filename.endswith(".png")):
            # Open the image and append it to the list
            images.append(Image.open(os.path.join(input_folder, filename)))
    
    # Save images as a GIF
    if ONLY_SBZ==False:
        output_path = input_folder+f'CL_plots_{res}_{t_res}_ALL_CLS.gif'
    if ONLY_SBZ==True:
        output_path = input_folder+f'CL_plots_{res}_{t_res}_ONLY_SBZS.gif'
    images[0].save(output_path,
                   save_all=True,
                   append_images=images[1:],
                   duration=100,  # Specify duration for each frame in milliseconds
                   loop=0)

In [33]:
if plotting==True:
    def convert_gif_to_mp4(input_file, output_file, fps,speed,bitrate='750k'):
        from moviepy.editor import VideoFileClip, vfx
        # Load the GIF file
        gif_clip = VideoFileClip(input_file)
    
        # Set the desired framerate if provided
        if fps:
            gif_clip = gif_clip.set_fps(fps)
        if speed != 1.0:
            gif_clip = gif_clip.fx(vfx.speedx, speed)
    
        # Write the GIF as an MP4 file
        gif_clip.write_videofile(output_file, codec="libx264",bitrate=bitrate)
    
    ONLY_SBZ=False
    # ONLY_SBZ=True
    input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_Tracking_Out/Plots/'
    if ONLY_SBZ==False:
        input_folder += 'ALL_CLS/'
    elif ONLY_SBZ==True:
        input_folder += 'ONLY_SBZS/'
    input_file = input_folder+f'CL_plots_{res}_{t_res}_ALL_CLS.gif'
    
    if ONLY_SBZ==False:
        output_file = f'/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_{res}_{t_res}_ALL_CLS.mp4'
    if ONLY_SBZ==True:
            output_file = f'/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_{res}_{t_res}_ONLY_SBZS.mp4'
        
    convert_gif_to_mp4(input_file, output_file, speed=0.25,fps=None)

Moviepy - Building video /mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_1km_1min_95nz_ALL_CLS.mp4.
Moviepy - Writing video /mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_1km_1min_95nz_ALL_CLS.mp4



Moviepy - Done !
Moviepy - video ready /mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_1km_1min_95nz_ALL_CLS.mp4
